# Final Results: GA-Optimized Fuzzy Wins on Holdout (PROOF-01)

End-to-end narrative a reviewer reads first: the simulated plant (01),
the classical baselines (02), the hand-tuned fuzzy (03), the coupled GA
fuzzy (04), the frozen scoreboard (05) — and the verdict below. Every
number on this page is RESTATED from frozen artifacts
(`results/comparison.csv`, `results/ga_optimized_params.json`, the 01-04
PNGs); nothing is recomputed and no training runs inline.



In [ ]:
import csv
import json
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src.fuzzy_controller import FUZZY_MFS, LABELS, tri
from src.ga_fuzzy_controller import load_optimized_mfs
from src.metrics import COLUMNS  # frozen column contract, never retyped
from src.weather_profile_generator import generate_weather_profile

print("labels:", LABELS)


In [ ]:
# Weather input: regenerate the seed-7 profile only if absent, then load.
# (Pattern-uniformity guard — 06 runs no simulation; artifacts are the only
# data source. Kept verbatim so the mechanical audit is uniform.)
CSV = "weather/weather_profile.csv"
if not os.path.exists(CSV):
    generate_weather_profile(CSV, seed=7)



In [ ]:
# Scoreboard load: stdlib csv only. Fail-closed provenance before any plot.
CSV_PATH = "results/comparison.csv"
with open(CSV_PATH) as f:
    rows = list(csv.DictReader(f))
assert list(rows[0].keys()) == COLUMNS, "stale artifact: header != COLUMNS"
assert len(rows) == 8, "expected 4 controllers x 2 weathers"
print("scoreboard: %d rows x %d cols" % (len(rows), len(COLUMNS)))
for r in rows:
    print("%-8s %-13s iae_T %6.2f iae_H %7.2f E %6.2f cost %8.2f rank %s"
          % (r["controller"], r["weather"], float(r["iae_T"]), float(r["iae_H"]),
             float(r["energy"]), float(r["cost"]), r["rank"]))



In [ ]:
# GA artifact load through the validated loader (fail-fast schema checks).
JSON = "results/ga_optimized_params.json"
with open(JSON) as _f:
    d = json.load(_f)
assert d["schema_version"] == 2 and d["population"] == 50 and d["generations"] == 50, \
    "stale artifact: expected run-3 schema v2 50x50"
mfs_opt, gains_opt = load_optimized_mfs(JSON)
print("schema v%s" % d.get("schema_version", 1),
      "| train fit %.2f" % d["fitness"], "| coupling", d.get("coupling"))
print("history gens", len(d["history"]["best"]))



In [ ]:
# Verdict RESTATEMENT: holdout costs reprinted from the scoreboard cell
# above with relative asserts only — no cost recomputation from histories.
# GA about 114.28, on/off about 513.75, PID about 645.26 (comments only).
holdout = [r for r in rows if r["weather"] == "holdout-seed21"]
costs = {r["controller"]: float(r["cost"]) for r in holdout}
assert costs["ga_fuzzy"] < costs["onoff"], costs
assert costs["ga_fuzzy"] < costs["pid"], costs
ranked = sorted(holdout, key=lambda r: int(r["rank"]))
print("holdout restated: GA %.2f < on/off %.2f AND GA < PID %.2f (rank %s)" %
      (costs["ga_fuzzy"], costs["onoff"], costs["pid"],
       next(r["rank"] for r in holdout if r["controller"] == "ga_fuzzy")))
print("verdict: GA < on/off AND GA < PID on holdout \u2014 METR-02 HOLDS")



In [ ]:
# Membership before/after: hand FUZZY_MFS solid vs GA-tuned dashed, drawn
# with the shared tri helper (same function the engine uses — no drift).
# Ranking strip on top restates the CSV rank column (stable: ties keep row order).
DISPLAY = {"onoff": "on/off", "pid": "PID", "fuzzy": "hand fuzzy", "ga_fuzzy": "GA fuzzy"}
fig = plt.figure(figsize=(12, 14))
gs = fig.add_gridspec(4, 2)
ra = fig.add_subplot(gs[0, :])
ra.barh([DISPLAY[r["controller"]] for r in ranked], [float(r["cost"]) for r in ranked])
ra.set_xlabel("holdout cost (restated from CSV)")
ra.set_title("Holdout cost ranking (CSV rank column)")
for i, r in enumerate(ranked):
    ra.text(float(r["cost"]), i, " #%s" % r["rank"], va="center", fontsize=9)
for j, key, lo, hi in [(0, "eT", -8, 8), (1, "eH", -40, 40)]:
    a = fig.add_subplot(gs[1, j])
    grid = np.linspace(lo, hi, 400)
    for c, w in zip(FUZZY_MFS[key]["centers"], FUZZY_MFS[key]["widths"]):
        a.plot(grid, [tri(x, c, w) for x in grid], "C0-", alpha=0.5)
    for c, w in zip(mfs_opt[key]["centers"], mfs_opt[key]["widths"]):
        a.plot(grid, [tri(x, c, w) for x in grid], "C1--", alpha=0.8)
    a.plot([], [], "C0-", label="hand %s" % key)
    a.plot([], [], "C1--", label="GA %s" % key)
    a.set_title("MF before/after: %s" % key)
    a.legend(fontsize=8)
fig.text(0.5, 0.52, "coupling k_heat/k_vent = %.2f/%.2f" % (gains_opt["k_heat"], gains_opt["k_vent"]),
         ha="center", fontsize=10)
print("overlay confirmed: hand solid vs GA dashed for eT and eH")



In [ ]:
# Prior-trail montage: the four sealed PNGs composited 2-by-2 via image
# reads (Agg-safe). Fallback: a text panel naming the trail order.
mgs = gs[2:, :].subgridspec(2, 2)
trail = ["notebooks/01_greenhouse_model.png", "notebooks/02_baselines.png",
         "notebooks/03_fuzzy.png", "notebooks/04_ga_optimization.png"]
try:
    imgs = [plt.imread(p) for p in trail]
    print("montage: 4 prior PNGs loaded")
except Exception as exc:
    imgs = []
    print("montage fallback (markdown links): %r" % exc)
for k, p in enumerate(trail):
    a = fig.add_subplot(mgs[k // 2, k % 2])
    if imgs:
        a.imshow(imgs[k])
    else:
        a.text(0.5, 0.5, "![trail](%s)" % p, ha="center")
    a.axis("off")
    a.set_title(p.split("/")[-1], fontsize=8)



**Closing claim:** the coupled GA fuzzy ranks #1 on the unseen seed-21
holdout on the locked 1.0/1.0/0.01 cost — below on/off AND below PID,
restated from the frozen scoreboard above. Saturation (`settling inf`,
RH 100% stretches) is structural across all controllers, not a GA defect
(02/03-note precedent): one-sided fogger authority with T-slaved sinks.



In [ ]:
fig.suptitle("Final results: GA-optimized fuzzy wins on holdout (PROOF-01)")
fig.tight_layout()
fig.savefig("notebooks/06_final_results.png", dpi=120)
plt.show()
print("saved notebooks/06_final_results.png")

